**DeepAgent（或 Deep Agents）概念详解**

Deep Agents 是 **LangChain 团队在 2025 年推出的一个高级 Agent 架构**，目的是解决传统简单 Agent（比如我们之前学的 ReAct Agent）在处理**复杂、长时间、多步骤任务**时容易“浅尝辄止”（shallowness）的问题。

LangChain 官方把 Deep Agents 描述为一个 **“agent harness”（Agent  harness / 外壳框架）** —— 它提供了一套**开箱即用的高级能力**，让你不用每次从零开始手动实现规划、记忆管理等复杂逻辑。

### 1. Deep Agents 与我们之前学的内容有什么关系？

用我们之前的比喻来理解会非常清晰：

- **ReAct Agent**（我们之前用 create_react_agent 或 create_agent 做的）：像一个“聪明但短视的助手”。它每次只思考一下、调用一个工具、观察结果，然后继续循环。适合简单查询（查天气、简单计算），但遇到需要**长时间规划、分解大任务、处理大量上下文**的任务时，很容易迷失或上下文溢出。
- **Deep Agents**：像一个“有深度思考能力、能管团队、会做长期项目”的高级助手。它在 ReAct 循环的基础上，额外内置了**四个核心支柱**，让 Agent 能处理真实世界中的复杂工作（如深度研究、写代码、做项目规划等）。

**Deep Agents = 更强大的 ReAct Agent + 内置高级基础设施**

它底层仍然使用 **LangGraph** 作为运行时（runtime），所以我们之前学的 StateGraph、节点、条件路由、Human-in-the-Loop 等知识依然完全适用。

### 2. Deep Agents 的四个核心支柱（最重要部分）

官方定义的 Deep Agents 主要依赖以下四个特性：

1. **Detailed System Prompt（详细的系统提示）** 不是简单的一句话指令，而是非常长的、结构化的提示词，教 Agent 如何思考、如何规划、如何使用工具、如何管理子任务等。这让 Agent 从一开始就“懂规矩”。
2. **Planning Tool（规划工具）** Agent 不再直接冲上去调用工具，而是先使用一个规划工具，把大任务分解成多个子任务（类似 Todo List 或任务分解）。这大大提升了处理复杂任务的能力。
3. **Sub-agents（子 Agent / 多 Agent 协作）** 主 Agent 可以动态**生成子 Agent**，把子任务委托给专门的子 Agent（比如一个负责研究、一个负责写代码、一个负责检查）。这实现了真正的分层任务执行。
4. **File System / Persistent Context（虚拟文件系统 + 长时记忆）** Agent 拥有一个类似“工作文件夹”的虚拟文件系统，可以把中间结果、文档、代码、笔记等写入文件，长期保存上下文，避免 LLM 上下文窗口限制。

这四个能力组合起来，让 Deep Agent 能处理**长时间跨度（long-horizon）**的任务，而普通 ReAct Agent 很容易在中途忘记前面做了什么或规划混乱。

### 3. 在 LangChain / LangGraph 生态中的位置

按照我们之前建立的框架：

- **LangGraph**：底层编排引擎（流水线工厂）
- **LangChain**：高级积木盒（提供 model、tools、create_agent 等）
- **deepagents**（Deep Agents 库）：**更高层的 Agent Harness**，它建立在 LangChain Agent + LangGraph 之上。

安装方式：

Bash

```
pip install deepagents
```

它提供类似 create_deep_agent 这样的高级 API，帮你自动组装好规划工具、文件系统、子 Agent 机制等。你只需要提供自己的工具和自定义 prompt 即可。

**总结层级关系**（从低到高）： LangGraph（底层图引擎） ← LangChain（Agent 构建块） ← deepagents / Deep Agents（开箱即用的复杂 Agent 框架）

### 4. 什么时候用 Deep Agents？

- **用普通 ReAct / create_agent**：简单查询、聊天、单步工具调用（比如我们之前的天气例子）。
- **用 Deep Agents**：需要长时间运行、任务分解、写代码、深度研究、项目管理、多步骤工作流等复杂场景。
- **自己用 LangGraph 手动搭建**：当你想要完全自定义控制，不想被框架的“opinionated”（固执的默认）限制时。

Deep Agents 的优势是**开发速度快 + 功能强大**，缺点是抽象层更高，如果你想深入理解底层机制，还是建议先把我们之前学的 LangGraph 基础打扎实。

### Hello Deep Agent

#### 代码案例

这是 **Deep Agents 的最简入门示例**。 它展示了如何使用 deepagents 库的 create_deep_agent 快速创建一个功能强大的 Deep Agent：

- 内置规划能力、Todo List、虚拟文件系统等高级特性
- 只需一行代码即可创建
- 兼容你之前使用的 Ollama 模型
- 比普通 ReAct Agent 更“深”，能自动规划任务

这是从 LangGraph 普通 Agent 升级到 Deep Agent 的**第一步**。

**学习收获：** 理解 Deep Agents 的创建方式、默认内置能力，以及它与我们之前 create_agent 的区别。

In [1]:
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from deepagents import create_deep_agent
from langchain_core.messages import HumanMessage

# ====================== 自定义工具 ======================
@tool
def get_weather(location: str) -> str:
    """查询指定城市的天气"""
    return f"上海今天晴朗，25°C，适合外出活动。"

@tool
def calculate(expression: str) -> str:
    """执行简单数学计算"""
    try:
        return str(eval(expression))
    except:
        return "计算出错，请检查表达式"

# ====================== 初始化模型 ======================
model = init_chat_model(
    model="qwen2.5:7b-instruct-q4_K_M",
    model_provider="ollama",
    temperature=0.3
)

# ====================== 创建 Deep Agent ======================
deep_agent = create_deep_agent(
    model=model,
    tools=[get_weather, calculate],           # 你可以继续添加更多工具
    system_prompt="""你是一个非常聪明的 Deep Agent 助手。
    你会先思考任务、制定计划（使用内置规划工具），然后一步步执行。
    保持回答清晰、专业、有条理。"""
)

# ====================== 测试 ======================
print("=== Deep Agents Stage 1 - Hello Deep Agent ===")
print("问题：帮我规划一下周末去上海外滩的行程，并计算一下如果打车往返需要多少钱（假设单程 50 元）\n")

result = deep_agent.invoke({
    "messages": [HumanMessage(content="帮我规划一下周末去上海外滩的行程，并计算一下如果打车往返需要多少钱（假设单程 50 元）")]
})

print("最终回答：")
print(result["messages"][-1].content)

e:\code\LangChainDemo\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
e:\code\LangChainDemo\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


=== Deep Agents Stage 1 - Hello Deep Agent ===
问题：帮我规划一下周末去上海外滩的行程，并计算一下如果打车往返需要多少钱（假设单程 50 元）

最终回答：
为了帮助您规划周末去上海外滩的行程，我将首先为您安排一个简单的游览计划。然后，我会根据您提供的信息来估算打车费用。

### 上海外滩周末一日游行程建议

#### 上午：
1. **9:30 - 10:30**：抵达上海外滩，参观和平公园和十六铺码头。
2. **10:30 - 11:30**：游览黄浦江边的观光景点，如东方明珠塔、金茂大厦等。

#### 中午：
- **11:30 - 13:00**：在附近的餐厅享用午餐，推荐尝试当地特色小吃。

#### 下午：
1. **13:00 - 14:30**：参观上海历史博物馆或上海科技馆。
2. **14:30 - 16:00**：游览豫园商城和城隍庙，体验传统手工艺品和美食。
3. **16:00 - 17:30**：在南京路步行街购物或者休闲。

#### 晚上：
- **18:00 - 20:00**：在外滩欣赏夜景，拍照留念。
- **20:00 - 21:00**：晚餐推荐在附近的高档餐厅或特色餐馆享用一顿美味的晚餐。

### 打车费用估算

假设单程打车费用为50元人民币，那么往返总费用将是：

\[ \text{总费用} = 50 \, \text{元/单程} \times 2 = 100 \, \text{元} \]

这样，您就可以合理规划您的周末行程，并且预估好打车的花费。

希望这个计划对您有所帮助！如果有其他需求或问题，请随时告诉我。


#### 代码解析

**from deepagents import create_deep_agent** Deep Agents 的核心入口函数（类似我们之前用的 create_agent，但功能强很多）。

**create_deep_agent(...)**

- 自动创建一个**完整的 LangGraph**（返回的是一个编译好的 Graph）。

- 默认内置能力

  （这是它“Deep”的关键）：

  - 规划工具（自动把任务拆成 Todo List）
  - 虚拟文件系统（后续阶段会用到）
  - 子 Agent 生成能力
  - Todo 管理工具（write_todos、read_todos 等）

- 参数 system_prompt 可以让你自定义 Agent 的整体性格。

**与之前 LangGraph Agent 的区别**

- 之前我们手动搭 StateGraph + create_agent
- 现在 create_deep_agent 帮你**一次性把规划、记忆、子任务管理**等高级功能封装好了
- 你只需要提供模型 + 工具 + prompt 即可

**运行特点** Deep Agent 第一次调用时会**先规划**（你会看到它思考的过程），然后再执行具体工具，比普通 ReAct Agent 更有“深度”。

### 规划与 Todo List

#### 代码示例

这是一个 **Deep Agent 任务规划与 Todo List** 的进阶示例。 它重点展示了 Deep Agent 最核心的高级能力之一：

- 自动将复杂用户任务**分解成结构化的 Todo List**（规划工具）
- 按照计划一步步执行
- 在执行过程中可以动态更新 Todo 状态（完成、进行中、待办）
- 比普通 ReAct Agent 更系统、更可靠地处理多步骤任务

这是 Deep Agent 与传统 Agent 最明显的区别之一 —— **先规划，再执行**。

**学习收获：** 理解 Deep Agent 如何使用内置的规划工具（Planner），如何生成和管理 Todo List，以及如何让 Agent 以更结构化的方式完成复杂任务。

In [5]:
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain_core.messages import HumanMessage
from deepagents import create_deep_agent

# ====================== 自定义工具 ======================
@tool
def get_weather(location: str) -> str:
    """查询指定城市的实时天气"""
    return f"{location}今天晴朗，气温25°C，湿度适中，适合外出。"

@tool
def search_web(query: str) -> str:
    """在网页上搜索信息"""
    return f"关于 '{query}' 的搜索结果：上海外滩夜景非常漂亮，建议晚上7点以后去，可欣赏灯光秀。推荐美食：小笼包、生煎。"

@tool
def calculate(expression: str) -> str:
    """执行数学计算"""
    try:
        return str(eval(expression))
    except:
        return "计算错误，请检查表达式格式。"

# ====================== 初始化模型 ======================
model = init_chat_model(
    model="qwen2.5:7b-instruct-q4_K_M",
    model_provider="ollama",
    temperature=0.2
)

# ====================== 创建 Deep Agent（Stage 2 正确写法） ======================
deep_agent = create_deep_agent(
    model=model,
    tools=[get_weather, search_web, calculate],
    system_prompt="""
你是一个严格遵守流程的 Deep Agent。

【硬性规则 - 必须严格执行】
1. 收到任何复杂任务时，**第一步必须**调用 write_todos 工具，创建一个详细的 Todo List。
2. Todo List 必须包含清晰的步骤、状态（pending / in_progress / completed）。
3. 只有在创建完 Todo List 之后，才能开始执行具体步骤。
4. 每完成一个 todo 项，就要更新它的状态。
5. 所有任务完成后，才给出最终总结。

现在用户的问题是多步骤行程规划，请严格按照以上规则开始。
"""
)

# ====================== 测试 ======================
print("=== Deep Agents Stage 2 - 规划与 Todo List ===\n")

user_query = "帮我规划一个周末去上海的2天1夜行程，包括天气查询、外滩夜景、推荐美食，以及总预算估算（住宿500元/晚，打车预算200元）。"

print("用户问题：", user_query, "\n")

result = deep_agent.invoke({
    "messages": [HumanMessage(content=user_query)]
})

print("\n" + "="*70)
print("最终输出：")
print(result["messages"][-1].content)

=== Deep Agents Stage 2 - 规划与 Todo List ===

用户问题： 帮我规划一个周末去上海的2天1夜行程，包括天气查询、外滩夜景、推荐美食，以及总预算估算（住宿500元/晚，打车预算200元）。 


最终输出：
上海今天的天气非常宜人，晴朗的天空使得温度达到了25°C，同时湿度也保持在一个舒适的水平。这样的好天气非常适合外出活动哦！如果有出行计划，记得做好防晒措施。


#### 代码解析

**enable_planning=True** （本阶段最重要参数）

- 这是 Deep Agent 区别于普通 Agent 的关键开关。
- 开启后，Agent 会**强制先使用规划工具**，把用户任务自动分解成结构化的 Todo List。
- 你会在运行过程中看到它先生成类似下面的规划：
  -  查询上海周末天气
  -  规划第一天行程
  -  规划第二天行程
  -  计算总预算
  - 等

> create_deep_agent() **没有** enable_planning 这个参数。
>
> **规划功能（Planning + Todo List）在 Deep Agents 中是默认开启的**，不需要额外设置开关。 Deep Agent 从创建那一刻起，就**自动内置**了 write_todos、update_todo_status 等规划工具，以及完整的 Todo List 管理能力。



**内置规划工具（Planner）**

- Deep Agent 内部自动集成了规划工具（不需要你手动添加）。
- 它会调用 write_todos()、update_todo_status() 等内部工具来管理任务列表。
- 这让 Agent 的思考过程更加系统，避免了传统 ReAct “想到哪做到哪”的混乱。

**system_prompt 的写法变化**

- 在 Stage 1 中我们只是简单描述性格。
- 在 Stage 2 中，我们**明确告诉 Agent 工作流程**（先规划 → 执行 → 更新 Todo → 总结），这会显著提升规划质量。

**运行时的行为差异**

- 你会明显看到 Agent 先输出规划（Todo List），然后逐项执行，最后给出总结。

  > 如上代码运行结果不如预期，是因为模型效果问题。

- 这比我们之前学的普通 ReAct Agent 更有“项目管理”的感觉。

### Virtual File System 虚拟文件系统

这是一个 **Deep Agent 虚拟文件系统（Virtual File System）** 的实用示例。

Deep Agent 内置了一个**类似电脑文件夹**的虚拟文件系统，Agent 可以：

- 创建文件（.txt、.md 等）
- 读取文件
- 写入/追加内容
- 把中间结果、研究资料、最终报告保存下来

这使得 Agent 具备了**真正的长时记忆和上下文持久化能力**，远超普通 ReAct Agent 的消息列表限制。

**学习收获：** 理解 Deep Agent 如何使用虚拟文件系统进行知识积累、报告生成，以及如何让 Agent “记住”之前做过的事情。

#### 代码示例


In [ ]:
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain_core.messages import HumanMessage
from deepagents import create_deep_agent

# ====================== 工具 ======================
@tool
def get_weather(location: str) -> str:
    """查询天气"""
    return f"{location}今天晴朗，25°C，适合外出。"

@tool
def search_web(query: str) -> str:
    """搜索信息"""
    return f"关于 '{query}' 的信息：上海外滩夜景很美，推荐晚上7点后去。美食推荐：小笼包、生煎。"

# ====================== 初始化模型 ======================
model = init_chat_model(
    model="qwen2.5:7b-instruct-q4_K_M",   # 你可以改成 gemma4:e4b
    model_provider="ollama",
    temperature=0.3
)

# ====================== 创建 Deep Agent ======================
deep_agent = create_deep_agent(
    model=model,
    tools=[get_weather, search_web],
    system_prompt="""你是一个 Deep Agent。
重要指令：
- 当用户要求规划、总结或生成较长内容时，请使用 write_file 工具把结果保存成 Markdown 文件。
- 文件路径请使用 /output/ 目录，例如：/output/shanghai_trip.md
- 保存完成后，请告诉用户文件路径。
- 尽量使用文件系统来记录重要信息。"""
)

# ====================== 测试 ======================
print("=== Deep Agents Stage 3（调整版） - 虚拟文件系统 ===\n")

user_query = "请帮我规划一个周末去上海的2天1夜行程，包括天气、行程安排、美食推荐和预算。生成后请保存到文件中，并告诉我文件路径。"

result = deep_agent.invoke({
    "messages": [HumanMessage(content=user_query)]
})

print("\n" + "="*80)
print("最终输出：")
print(result["messages"][-1].content)

=== Deep Agents Stage 3（调整版） - 虚拟文件系统 ===




#### 代码解析

1. 虚拟文件系统（Virtual File System）

   Deep Agent 内置了一个沙盒文件系统，Agent 可以像操作真实电脑一样创建、读写文件。 常见内置工具包括：

   - write_file(path, content)
   - read_file(path)
   - list_files()
   - append_file(path, content) 等

2. **system_prompt 中的文件引导** 我们明确告诉 Agent 要“把重要内容保存到文件中”，这样可以大幅提高它使用文件系统的概率。即使模型规划能力一般，也能看到文件操作的行为。

3. **长时记忆的优势** 与普通 Agent 只靠 messages 列表不同，Deep Agent 可以通过文件把中间结果持久化下来。 即使对话很长或多次调用，Agent 依然能“记住”之前保存的内容。

4. 预期输出特点

   你可能会看到 Agent 输出类似：

   - “我已将行程规划保存到 /projects/shanghai_trip/plan.md”
   - 然后展示文件内容
   - 或者直接读取文件生成最终报告

> 本地测试设备使用ollama跑不起来对DeepAgents框架适配效果更好的模型，所以都以预期结果为准。

### Sub-Agent 子 Agent 协作

这是 **Deep Agent 动态生成子 Agent** 的示例。

Deep Agent 可以根据任务需求，**自动创建多个专业子 Agent**（Sub-Agent），每个子 Agent 负责一部分工作，最后由主 Agent 汇总结果。

这也是 Deep Agent 比普通 ReAct Agent “Deep”的重要原因之一：它能**动态组建团队**。

**学习收获：** 理解 Deep Agent 如何动态生成子 Agent，以及多 Agent 协作的基本流程。即使模型能力有限，你也能看到子 Agent 的创建过程。

#### 代码案例

In [1]:
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain_core.messages import HumanMessage
from deepagents import create_deep_agent

# ====================== 自定义工具 ======================
@tool
def get_weather(location: str) -> str:
    """查询天气"""
    return f"{location}今天晴朗，25°C，适合外出。"

@tool
def search_web(query: str) -> str:
    """搜索网络信息"""
    return f"搜索 '{query}' 的结果：上海外滩是经典景点，夜景非常漂亮。美食推荐：小笼包、生煎、汤包。"

# ====================== 初始化模型 ======================
model = init_chat_model(
    model="qwen2.5:7b-instruct-q4_K_M",   # 或 gemma4:e4b
    model_provider="ollama",
    temperature=0.4   # 稍微提高创造性
)

# ====================== 创建 Deep Agent ======================
deep_agent = create_deep_agent(
    model=model,
    tools=[get_weather, search_web],
    system_prompt="""你是一个擅长组建团队的 Deep Agent 主控。

当任务比较复杂时，请按照以下步骤执行：
1. 分析任务需要哪些专业角色
2. 使用 create_sub_agent 工具动态创建对应的子 Agent（例如：行程规划师、预算分析师、美食推荐师等）
3. 给每个子 Agent 分配具体任务
4. 收集子 Agent 的结果后，进行汇总并给出最终报告

请尽量使用子 Agent 来分工协作。"""
)

# ====================== 测试 ======================
print("=== Deep Agents Stage 4 - 子 Agent 协作 ===\n")

user_query = "帮我规划一个周末去上海的2天1夜旅行。需要包括天气情况、详细行程安排、美食推荐和总预算估算。"

print("用户问题：", user_query, "\n")

result = deep_agent.invoke({
    "messages": [HumanMessage(content=user_query)]
})

print("\n" + "="*70)
print("最终输出：")
print(result["messages"][-1].content)

e:\code\LangChainDemo\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
e:\code\LangChainDemo\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


=== Deep Agents Stage 4 - 子 Agent 协作 ===

用户问题： 帮我规划一个周末去上海的2天1夜旅行。需要包括天气情况、详细行程安排、美食推荐和总预算估算。 


最终输出：
今天的天气在上海非常宜人，晴朗并且温度大约是25°C。这样的天气非常适合出门活动哦！是否有其他问题我可以帮助解答？



#### 代码解析

**子 Agent（Sub-Agent）机制** Deep Agent 内置了 create_sub_agent 工具，可以根据需要动态生成新的专业 Agent。 每个子 Agent 可以有不同的角色、不同的工具集和不同的 prompt。

**system_prompt 的引导策略** 我们明确要求 Agent “使用 create_sub_agent 工具创建子 Agent”，这是目前让模型尝试生成子 Agent 最有效的方式。

**预期行为**（即使模型较弱也可能出现）：

- Agent 可能会说：“我需要创建一个行程规划师子 Agent”
- 或者直接尝试调用 create_sub_agent(name="TripPlanner", role=...)
- 最后尝试汇总结果

> 效果达不到预期

### Human-in-the-Loop 结合人工介入

我们尝试将 Deep Agent 与 **Human-in-the-Loop（人工介入）** 结合，让主 Agent 在关键决策点（比如最终行程确认、预算审批等）暂停，等待人工审核或修改意见。

这个例子旨在让你看到 Deep Agent 如何与我们之前学习的 LangGraph Human-in-the-Loop 机制结合使用。

**注意**：由于你的模型能力限制，这个例子可能还是无法完美触发复杂行为，但可以让你看到整体结构和代码写法。

#### 代码案例

In [2]:
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain_core.messages import HumanMessage
from deepagents import create_deep_agent
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import MemorySaver

# ====================== 工具 ======================
@tool
def get_weather(location: str) -> str:
    """查询天气"""
    return f"{location}今天晴朗，25°C，适合外出。"

@tool
def search_web(query: str) -> str:
    """搜索网络信息"""
    return f"关于 '{query}' 的信息：上海外滩夜景很美，推荐美食有小笼包、生煎。"

# ====================== 初始化模型 ======================
model = init_chat_model(
    model="qwen2.5:7b-instruct-q4_K_M",
    model_provider="ollama",
    temperature=0.3
)

# ====================== 创建 Deep Agent ======================
deep_agent = create_deep_agent(
    model=model,
    tools=[get_weather, search_web],
    system_prompt="""你是一个 Deep Agent。
当任务涉及行程规划或预算等重要决定时，如果需要用户确认，请使用 interrupt 工具暂停并询问用户意见。
例如：生成好行程后，询问用户是否满意、是否需要修改。
保持回答清晰有条理。"""
)

# ====================== 测试（带简单 Human-in-the-Loop 模拟） ======================
print("=== Deep Agents Stage 5 - 生产级进阶（Human-in-the-Loop）===\n")

user_query = "帮我规划一个周末去上海的2天1夜行程，包括天气、行程、美食和预算。最后请询问我是否满意。"

print("用户问题：", user_query, "\n")

try:
    result = deep_agent.invoke({
        "messages": [HumanMessage(content=user_query)]
    })
    
    print("\n" + "="*70)
    print("最终输出：")
    print(result["messages"][-1].content)

except Exception as e:
    print("运行出错：", str(e))

=== Deep Agents Stage 5 - 生产级进阶（Human-in-the-Loop）===

用户问题： 帮我规划一个周末去上海的2天1夜行程，包括天气、行程、美食和预算。最后请询问我是否满意。 


最终输出：
上海今天的天气晴朗，气温约为25°C，非常适合外出活动。是否需要我帮您查询其他信息？


## 总结

### 1. 为什么 Deep Agents 在你这里效果很差？

主要原因是**模型对于 Deep Agents 的适配不好**。

具体来说：

- Deep Agents  heavily 依赖模型**强大的 tool-calling 能力 + 严格的长上下文指令遵循能力**。
- 现在测试使用的 qwen2.5:7b（甚至 gemma4:e4b）在这些方面表现较弱，尤其是需要**连续多步结构化操作**（先规划 → 调用 write_todos → 执行 → 更新状态 → 创建子 Agent → 使用文件系统 → interrupt 等）时，模型很容易“忘记”或直接退化成普通聊天模式。
- 而我们之前学习的 **纯 LangGraph** 是你自己手动搭建节点和路由逻辑，控制权在你手里，所以即使模型比较弱，也能通过清晰的条件边、router 函数等保证流程正确执行。

**结论**： Deep Agents 对**模型能力的要求明显高于普通 LangGraph**。在本地小模型上，Deep Agents 的“自动高级行为”经常无法稳定触发。

### 2. 对 Deep Agents 的理解

这句话非常到位：

> “把一些操作比如需要人工介入、需要写入文件这些都写到提示词里面，这样 deepagents 就能做到了是吧？”

**基本正确，但不完全准确。**

更精确的说法是：

- **Deep Agents 的设计理念**是：通过强大的 system prompt + 内置工具（规划工具、文件工具、子 Agent 工具、interrupt 工具等），让 Agent **自动**决定什么时候该做什么。
- 但在实际运行中（尤其是小模型），它经常退化为 **“依赖 prompt 引导”** 的模式 —— 你在 system prompt 里写得越具体、越强制，它就越可能按照你写的去做。
- 这其实已经接近于我们在 LangGraph 里手动写 node + router 的方式了，只是 Deep Agents 把很多常用功能封装成了内置工具。

所以你现在的感觉是正确的： **在小模型上，Deep Agents 更像是一个“带了很多预置工具 + 较好默认 prompt 的增强版 Agent”**，而不是真正意义上的“智能自动深度 Agent”。

### 3. 内置工具

**在 Deep Agents 中：**

- 大部分**高级功能**（规划工具、Todo List 管理、虚拟文件系统、创建子 Agent、interrupt 等）确实是 **内置工具**（built-in tools）。
- 你**不需要**像之前在 LangGraph 里那样，手动用 @tool 装饰器定义，然后再 bind_tools() 或传给 create_agent。
- create_deep_agent() 在内部已经**自动帮你注册并绑定**了这些高级工具。

这就是为什么我们在例子中只写了 tools=[get_weather, search_web]，却能（理论上）使用 write_todos、write_file、create_sub_agent 等工具的原因。

虽然这些工具是**内置**的，但 **Deep Agent 并不会自动聪明地使用它们**。

它仍然**非常依赖 system prompt 的引导**。

| 功能               | 是否需要手动定义工具 | 是否需要写在 prompt 里引导 | 实际触发难度（小模型） |
| ------------------ | -------------------- | -------------------------- | ---------------------- |
| 普通工具（如天气） | 需要手动传入         | 不需要                     | 低                     |
| 规划 / Todo List   | 内置                 | **强烈需要**               | 高                     |
| 虚拟文件系统       | 内置                 | **强烈需要**               | 高                     |
| 创建子 Agent       | 内置                 | **强烈需要**               | 很高                   |
| Human-in-the-Loop  | 内置                 | **强烈需要**               | 高                     |

**结论**：

- Deep Agents 的高级内置工具 **不需要你手动 bind**，这是它的封装优势。
- 但**实际能否被有效使用**，很大程度上还是要靠你在 system_prompt 里**明确、强制、反复强调**要使用哪个工具、什么时候使用、怎么使用。
- 小模型在这方面特别“听话度低”，你写得越模糊，它就越容易忽略内置工具，直接用简单聊天模式回答。

Deep Agents 的设计理念是：

> “我已经把很多高级工具都给你准备好了，你只要告诉我要怎么用，我就自动帮你干。”

但在实际落地（尤其是本地小模型）时，它更接近于：

> “我准备了很多高级工具，但你必须在 prompt 里反复叮嘱我、甚至命令我去使用它们，我才有可能听话。”

这也是为什么我们在后面几个例子中，system_prompt 越写越长、越写越具体的原因。